In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn scipy

In [ ]:
# 1. Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

from scipy.cluster.hierarchy import dendrogram, linkage


# 2. Generate Synthetic Wholesale Customers Dataset

X, _ = make_blobs(
    n_samples=200,
    n_features=6,
    centers=4,
    cluster_std=1.5,
    random_state=42
)


# 3. Convert Values into Positive Spending Amounts

minmax = MinMaxScaler(feature_range=(100, 10000))

X_scaled = minmax.fit_transform(X)

df = pd.DataFrame(
    X_scaled,
    columns=[
        "Fresh",
        "Milk",
        "Grocery",
        "Frozen",
        "Detergents_Paper",
        "Delicassen"
    ]
)

df = df.round(2)

print("Generated Wholesale Customers Dataset:")
print(df.head())


# 4. Save Dataset as CSV

df.to_csv("wholesale_customers.csv", index=False)

print("\nDataset saved successfully!")


# 5. Explore Dataset

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())


# 6. Visualize Customer Spending

plt.figure(figsize=(10, 6))

df.boxplot()

plt.title("Customer Spending Distribution")
plt.ylabel("Annual Spending")
plt.xticks(rotation=30)
plt.show()


# 7. Correlation Analysis

correlation = df.corr()

plt.figure(figsize=(9, 6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.show()


# 8. Standardize the Dataset

scaler = StandardScaler()

X_standardized = scaler.fit_transform(df)


# 9. Create Linkage Matrix Using Ward Method

linkage_matrix = linkage(
    X_standardized,
    method="ward",
    metric="euclidean"
)


# 10. Plot Dendrogram

plt.figure(figsize=(12, 6))

dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=30,
    leaf_rotation=90,
    leaf_font_size=10
)

plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Customer Groups")
plt.ylabel("Euclidean Distance")
plt.show()


# 11. Apply Agglomerative Hierarchical Clustering

model = AgglomerativeClustering(
    n_clusters=4,
    metric="euclidean",
    linkage="ward"
)

clusters = model.fit_predict(X_standardized)


# 12. Add Cluster Labels to Dataset

df["Cluster"] = clusters

print("\nClustered Dataset:")
print(df.head(10))


# 13. Count Customers in Each Cluster

print("\nCustomers in Each Cluster:")

print(df["Cluster"].value_counts().sort_index())


# 14. Calculate Cluster-wise Average Spending

cluster_analysis = df.groupby("Cluster").mean(numeric_only=True)

print("\nAverage Spending of Each Cluster:")
print(cluster_analysis)


# 15. Evaluate Clustering Using Silhouette Score

score = silhouette_score(
    X_standardized,
    clusters,
    metric="euclidean"
)

print("\nSilhouette Score:", score)


# 16. Visualize Clusters Using First Two Features

plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="Fresh",
    y="Grocery",
    hue="Cluster",
    palette="Set1",
    s=60
)

plt.title("Hierarchical Clustering of Wholesale Customers")
plt.xlabel("Fresh Spending")
plt.ylabel("Grocery Spending")
plt.show()


# 17. Save Clustered Dataset

df.to_csv("wholesale_customers_clustered.csv", index=False)

print("\nClustered dataset saved successfully!")